In [1]:
import anndata as ad
import scipy.sparse as sp
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from infer_wscreni import GenePeakOverlapLabs
from infer_wscreni import infer_wScReNI_sc_networks

In [2]:
DATASET = "retinal"   # swap to "pbmc", "seaad_paired", "seaad_unpaired"
BASE    = f"../../../data/processed/{DATASET}"

# 1. Load files
rna      = ad.read_h5ad(f"{BASE}_rna_sub.h5ad")       # (400 cells, 500 genes)
knn      = np.load(f"{BASE}_knn_indices.npy")          # (400, 20)  0-based
triplets = pd.read_csv(f"{BASE}_triplets.csv")         # target_gene | peak | spearman_r | TF
labels   = pd.read_csv(f"{BASE}_gene_labels.csv")      # gene | type | associated_peaks | associated_TFs


### If you already have the `retinal_with_networks.h5ad` file you can skip until the loading of the h5ad file

In [4]:
peak_matrix   = np.load(f"../../../data/processed/peak_matrix.npy")

In [5]:
X = rna.X.toarray() if sp.issparse(rna.X) else rna.X  # (400, 500)
expr_df = pd.DataFrame(
    X.T,                        # → (500 genes, 400 cells)
    index   = rna.var_names,    # 500 gene names
    columns = rna.obs['_original_rna_cell'],    # 400 cell barcodes
)

In [6]:
# 3. peak_df: peaks × cells
# peak_matrix is (400 cells, 217 peaks) — transpose to (217 peaks, 400 cells)
# Peak names come from unique peaks in triplets (217 unique values)
peak_names = triplets["peak"].unique()                 # 217 peak names
peak_df = pd.DataFrame(
    peak_matrix.T,              # → (217 peaks, 400 cells)
    index   = peak_names,
    columns = rna.obs_names,
)

In [7]:
rows = []
for _, row in labels.iterrows():
    gene  = row["gene"]
    gtype = row["type"]          # "TF" or "target"

    if pd.isna(row["associated_peaks"]):
        # Remove empt peak rows
        continue
    else:
        peaks_for_gene = str(row["associated_peaks"]).split(",")
        TF_str = (
            ";".join(str(row["associated_TFs"]).split(","))
            if pd.notna(row["associated_TFs"]) else ""
        )
        for peak in peaks_for_gene:
            rows.append({"gene": gene, "peak": peak.strip(), "TF": TF_str, "label": gtype})

labs_df = pd.DataFrame(rows)

In [8]:
labs = GenePeakOverlapLabs(
    genes  = labs_df["gene"].tolist(),
    peaks  = labs_df["peak"].tolist(),
    TFs    = labs_df["TF"].tolist(),      # semicolon-separated TF names
    labels = labs_df["label"].tolist(),   # "TF" or "target"
)

# # Sanity checks
# assert expr_df.shape == (500, 400), f"Unexpected expr shape: {expr_df.shape}"
# assert peak_df.shape == (217, 400), f"Unexpected peak shape: {peak_df.shape}"
# assert knn.shape     == (400, 20),  f"Unexpected KNN shape: {knn.shape}"

print(f"expr_matrix : {expr_df.shape}  (genes × cells)")
print(f"peak_df     : {peak_df.shape}  (peaks × cells)")
print(f"knn         : {knn.shape}  (cells × neighbors)")
print(f"labs entries: {len(labs.genes)} gene-peak rows")
print(f"TF genes    : {(labels['type'] == 'TF').sum()}, "
        f"target genes: {(labels['type'] == 'target').sum()}")

expr_matrix : (500, 400)  (genes × cells)
peak_df     : (217, 400)  (peaks × cells)
knn         : (400, 20)  (cells × neighbors)
labs entries: 228 gene-peak rows
TF genes    : 44, target genes: 456


In [9]:
# Run wScReNI
networks = infer_wScReNI_sc_networks(
    expr_matrix              = expr_df,
    gene_peak_overlap_matrix = peak_df,
    gene_peak_overlap_labs   = labs,
    nearest_neighbors_idx    = knn,       # (400, 20), 0-based
    network_path             = "output/networks",
    data_name                = DATASET,
    cell_index               = None,      # None = all 400 cells
    nthread                  = 8,
    max_cell_per_batch       = 10,
)


Total number of cells: 400
Cell 0 to cell 9
Cell 10 to cell 19
Cell 20 to cell 29
Cell 30 to cell 39
Cell 40 to cell 49
Cell 50 to cell 59
Cell 60 to cell 69
Cell 70 to cell 79
Cell 80 to cell 89
Cell 90 to cell 99
Cell 100 to cell 109
Cell 110 to cell 119
Cell 120 to cell 129
Cell 130 to cell 139
Cell 140 to cell 149
Cell 150 to cell 159
Cell 160 to cell 169
Cell 170 to cell 179
Cell 180 to cell 189
Cell 190 to cell 199
Cell 200 to cell 209
Cell 210 to cell 219
Cell 220 to cell 229
Cell 230 to cell 239
Cell 240 to cell 249
Cell 250 to cell 259
Cell 260 to cell 269
Cell 270 to cell 279
Cell 280 to cell 289
Cell 290 to cell 299
Cell 300 to cell 309
Cell 310 to cell 319
Cell 320 to cell 329
Cell 330 to cell 339
Cell 340 to cell 349
Cell 350 to cell 359
Cell 360 to cell 369
Cell 370 to cell 379
Cell 380 to cell 389
Cell 390 to cell 399


In [10]:
# 7. Output
# networks is a plain list of length 400
# networks[i] is a np.ndarray of shape (500, 500) for cell i
# networks[i][row, col] = regulatory weight of gene col → gene row in cell i
#
# Files are also saved to: output/networks/wScReNI/{i}.{cell_barcode}.network.txt

# Optional: convert to dict keyed by cell barcode for easier lookup
cell_barcodes = rna.obs_names.tolist()
networks_dict = {cell_barcodes[i]: networks[i] for i in range(len(networks))}

# Optional: save results back into AnnData
rna.uns["wScReNI_networks"] = networks
rna.write_h5ad(f"output/{DATASET}_with_networks.h5ad")

print(f"\nDone. {len(networks)} networks inferred.")
print(f"Network shape per cell : {networks[0].shape}")   # (500, 500)
print(f"Output dir             : output/networks/wScReNI/")


Done. 400 networks inferred.
Network shape per cell : (500, 500)
Output dir             : output/networks/wScReNI/


### h5ad file loading

In [3]:
H5AD_FILE = f"output/{DATASET}_with_networks.h5ad"
GROUND_TRUTH_FILE = "../../../data/processed/mmp9.TSV.5kb_TF_target.df.txt"

In [4]:
ground_truth_df = pd.read_csv(GROUND_TRUTH_FILE,
    sep="\t")

ground_truth_df.dropna()

tf_target_reference = set(
    ground_truth_df["TF"].astype(str) + "_" + ground_truth_df["Target_genes"].astype(str)
)

print("Reference edges:", len(tf_target_reference))

Reference edges: 4185048


In [5]:
adata = ad.read_h5ad(H5AD_FILE)

networks = adata.uns["wScReNI_networks"]
genes = adata.var_names.tolist()
cells = adata.obs_names.tolist()

n_genes = len(genes)

print("Cells:", len(networks))
print("Genes:", n_genes)

gene_to_idx = {g: i for i, g in enumerate(genes)}

Cells: 400
Genes: 500


In [6]:
networks_df = [
    pd.DataFrame(net, index=genes, columns=genes)
    for net in networks
]

In [7]:
from calculate_scnetwork_precision_recall import calculate_scnetwork_precision_recall
from calculate_scnetwork_precision_recall_top import calculate_scnetwork_precision_recall_top


scNetworks = {
    "CSN": networks_df,
    "wScReNI": networks_df
}

results = calculate_scnetwork_precision_recall(
    scNetworks=scNetworks,
    TF_target_pair=tf_target_reference,
    gene_id_gene_name_pair=None,
    gene_name_type=None
)

[START] precision/recall computation

[INFO] Cutoff = 0 | elapsed = 0.28s
  [NET] CSN
  [NET] wScReNI

[INFO] Cutoff = 1000 | elapsed = 194.88s
  [NET] wScReNI


In [8]:
for k, df in results.items():
    print("\nTOP K =", k)
    print(df.groupby("scNetwork_type")[["precision", "recall"]].mean())


TOP K = 0
                precision    recall
scNetwork_type                     
CSN              0.045446  0.249912
wScReNI          0.045446  0.249912

TOP K = 1000
                precision    recall
scNetwork_type                     
wScReNI           0.03693  0.006264


C:\Users\Edi\AppData\Local\Temp\ipykernel_41772\2657440475.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby("scNetwork_type")[["precision", "recall"]].mean())
C:\Users\Edi\AppData\Local\Temp\ipykernel_41772\2657440475.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby("scNetwork_type")[["precision", "recall"]].mean())
